# maniol-elvish 3 階層フィルタリング済みデータ監査・微調整ワークベンチ（`dataset-audit-001`）

手作業で仕上げた **3 階層フォルダ（`向き / 画角 / アングル(__姿勢)`）** の中から、**入れ間違い・漏れ・フォルダ内での外れ値・別フォルダ間重複** を自動炙り出し、細かなデータ調整を行うためのノートブックです。

### 処理の流れ
1. **セットアップ & モデル初期化**: `rtmlib` + `WD14-Tagger v3` + `DINOv2`（GPU クリーンインストール対応）
2. **3 階層フォルダのスキャン**: 全フォルダの枚数・構成・完全同一ファイル重複を確認
3. **1 パス特徴量抽出＋キャッシュ**: 初回のみ推論し `features_cache.npz` に保存（2 回目以降は **1 秒で再読み込み** でき、フォルダ移動後も再推論不要！）
4. **4 軸 5-Fold OOF Classifier 監査**:
   - `orientation` / `framing` / `camera_angle` / `posture` の 4 軸で 5-Fold 交差検証（Out-of-Fold 予測）を実行
   - **「現フォルダのラベル」と「Classifier が高確率で予測した別ラベル」が食い違う画像** ＝ 入れ間違い最有力候補を優先度順にランキング
   - 葉フォルダ内の DINOv2 外れ値・別フォルダ間のニア重複・明らかな骨格矛盾も同時検出
5. **サムネイル確認 & 安全なフォルダ移動**: 怪しい候補を軸別に目視確認し、承認した画像だけ一括移動（`dry_run=True` 確認付き）→ 移動後はキャッシュ再利用で 1 秒再監査！

## 1. セットアップ

In [ ]:
# CPU 版 onnxruntime の混入を防ぎつつ scikit-learn を含めて導入
%pip install -q rtmlib opencv-python huggingface_hub transformers pillow pandas matplotlib scikit-learn
%pip uninstall -y onnxruntime onnxruntime-gpu
%pip install -q --no-cache-dir onnxruntime-gpu

In [ ]:
!nvidia-smi

## 2. Google Drive マウント & 共通モジュール読み込み

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/belre/stable-diffusion-pipelines.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/stable-diffusion-pipelines")

local_inspector_root = Path("..").resolve()
if (local_inspector_root / "common" / "__init__.py").exists():
    INSPECTOR_ROOT = local_inspector_root
else:
    if not REPO_DIR.exists():
        !git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull
    INSPECTOR_ROOT = REPO_DIR / "colab" / "snapshot-inspector"

if str(INSPECTOR_ROOT) not in sys.path:
    sys.path.insert(0, str(INSPECTOR_ROOT))

import os
import pandas as pd

from common import (
    DEFAULT_DINOV2_REPO,
    DEFAULT_WD14_REPO,
    DINOv2Embedder,
    WD14Tagger,
    apply_relabel_moves,
    archive_folder_to_drive,
    discover_labeled_dataset,
    extract_or_load_dataset_features,
    get_onnx_device_config,
    init_wholebody,
    run_dataset_audit,
    save_results_to_drive,
    show_audit_suspects,
)

rtmlib_device, ort_providers = get_onnx_device_config()
print(f"INSPECTOR_ROOT: {INSPECTOR_ROOT}")
print(f"ONNX device: {rtmlib_device} / providers: {ort_providers}")

## 3. 3 階層フィルタリング済みデータの配置 & 構成確認

手作業で仕分けた 3 階層フォルダ（`向き / 画角 / アングル(__姿勢)`）を `./data/maniol-elvish/curated_v1/` へ展開・配置します。

In [ ]:
CHARACTER_NAME = "maniol-elvish"
DATASET_ROOT = f"./data/{CHARACTER_NAME}/curated_v1"
AUDIT_OUTPUT_DIR = f"./output/{CHARACTER_NAME}/audit_001"
CACHE_DIR = f"{AUDIT_OUTPUT_DIR}/cache"

!mkdir -p {DATASET_ROOT} {AUDIT_OUTPUT_DIR} {CACHE_DIR}

# Google Drive から ZIP 展開またはコピーする例（環境に合わせてコメント解除・編集してください）
# !unzip -q '/content/drive/MyDrive/Colab Notebooks/maniol-elvish/curated_v1.zip' -d {DATASET_ROOT}/

dataset_df = discover_labeled_dataset(DATASET_ROOT)
print(f"総画像数: {len(dataset_df)} 枚 | 葉フォルダ数: {dataset_df['leaf_path'].nunique() if len(dataset_df) else 0}")

if len(dataset_df) > 0:
    print("\n=== 向き (label_orientation) × 画角 (label_framing) ===")
    display(pd.crosstab(dataset_df["label_orientation"], dataset_df["label_framing"], margins=True))
    print("\n=== アングル (label_camera_angle) × 姿勢 (label_posture) ===")
    display(pd.crosstab(dataset_df["label_camera_angle"], dataset_df["label_posture"], margins=True))
    n_exact_dup = int((dataset_df["exact_dup_count"] > 1).sum())
    if n_exact_dup > 0:
        print(f"⚠️ 内容完全一致（MD5重複）の画像が {n_exact_dup} 枚あります。")

## 4. 1 パス特徴量抽出（キャッシュ対応）

初回のみ 3 モデル（`rtmlib` + `WD14-Tagger v3` + `DINOv2`）をロードして特徴量を抽出し、`CACHE_DIR` に保存します。  
**2 回目以降（ファイル移動後の再監査など）はキャッシュから 1 秒で読み込まれます**（モデル初期化すら不要）。

In [ ]:
FORCE_RECOMPUTE = False  # キャッシュを作り直したいときだけ True

cache_ready = (
    not FORCE_RECOMPUTE
    and os.path.exists(os.path.join(CACHE_DIR, "features_cache.npz"))
    and os.path.exists(os.path.join(CACHE_DIR, "features_table.csv"))
)

if cache_ready:
    wholebody = wd14_tagger = dinov2_embedder = None
else:
    wholebody = init_wholebody(mode="performance", device=rtmlib_device)
    wd14_tagger = WD14Tagger(repo_id=DEFAULT_WD14_REPO, providers=ort_providers)
    dinov2_embedder = DINOv2Embedder(model_id=DEFAULT_DINOV2_REPO)

feat_df, dino_embs, aux_feats, aux_names = extract_or_load_dataset_features(
    dataset_df=dataset_df,
    cache_dir=CACHE_DIR,
    pose_model=wholebody,
    tagger=wd14_tagger,
    embedder=dinov2_embedder,
    force_recompute=FORCE_RECOMPUTE,
)
print(f"特徴量シェイプ: DINOv2={dino_embs.shape}, Aux={aux_feats.shape}")

## 5. 4 軸 5-Fold OOF Classifier 監査（入れ間違い・外れ値・重複の自動検出）

各軸の交差検証スコア（Accuracy / Macro-F1）と混同行列を確認しつつ、**「入れ間違い・漏れ候補（`suspects_df`）」** を優先度順に出力します。

In [ ]:
# 監査の感度設定（候補が多すぎたら suspect_prob_thr を 0.80 に上げ、厳しく見たいなら 0.60 に下げる）
SUSPECT_PROB_THR = 0.70       # OOF 予測が別クラスをこの確率以上で推したら入れ間違い候補
OUTLIER_Z_THR = 2.5           # 同一葉フォルダ内で DINOv2 類似度が平均から何σ低ければ外れ値とするか
CROSS_LEAF_DUP_SIM = 0.985    # 別フォルダ間でのほぼ同一画像（重複配置）検知しきい値

audited_df, suspects_df, axis_reports = run_dataset_audit(
    feat_df=feat_df,
    dino_embs=dino_embs,
    aux_feats=aux_feats,
    n_splits=5,
    suspect_prob_thr=SUSPECT_PROB_THR,
    outlier_z_thr=OUTLIER_Z_THR,
    cross_leaf_dup_sim=CROSS_LEAF_DUP_SIM,
)

# 結果を CSV 保存
AUDITED_CSV = os.path.join(AUDIT_OUTPUT_DIR, "audited_dataset.csv")
SUSPECTS_CSV = os.path.join(AUDIT_OUTPUT_DIR, "audit_suspects.csv")
audited_df.to_csv(AUDITED_CSV, index=False, encoding="utf-8-sig")
suspects_df.to_csv(SUSPECTS_CSV, index=False, encoding="utf-8-sig")
print(f"📄 保存完了: {SUSPECTS_CSV} ({len(suspects_df)} 件)")

# 各軸の混同行列を表示（どのフォルダ同士が混ざりやすいか一目で確認）
for axis_name, rep in axis_reports.items():
    if not rep["skipped"]:
        print(f"\n=== Confusion Matrix: {axis_name} (acc={rep['accuracy']}, macro_f1={rep['macro_f1']}) ===")
        display(rep["confusion_matrix"])

display(suspects_df.head(20))

## 6. 怪しい候補の軸別サムネイルレビュー

優先度（`priority`）が高い順に、軸ごとの入れ間違い候補・葉内 DINOv2 外れ値をサムネイルで目視確認します。

In [ ]:
# 1. 向き (orientation) の入れ間違い最有力候補
show_audit_suspects(suspects_df, filter_keyword="oof_orientation", n=12, cols=4)

# 2. 画角 (framing) の入れ間違い最有力候補
show_audit_suspects(suspects_df, filter_keyword="oof_framing", n=12, cols=4)

# 3. アングル (camera_angle) の入れ間違い最有力候補
show_audit_suspects(suspects_df, filter_keyword="oof_camera_angle", n=12, cols=4)

# 4. 姿勢 (posture) の入れ間違い最有力候補
show_audit_suspects(suspects_df, filter_keyword="oof_posture", n=12, cols=4)

# 5. 葉フォルダ内 DINOv2 外れ値（服装崩れ・破綻・誤配置の拾い漏れ候補）
show_audit_suspects(suspects_df, filter_keyword="leaf_visual_outlier", n=12, cols=4)

## 7. 承認した画像の安全移動（再配置）& Drive バックアップ

確認して移動したい画像は、`audit_suspects.csv` の `approve_move` 列を `True` にする（または下のセルで `APPROVE_FILES` にファイル名を指定する）ことで安全に移動できます。  
移動後に Section 3 → 4（キャッシュ 1 秒読み込み）→ 5 を再実行すれば、**移動によって交差検証スコアがどれだけ上がったか**をすぐ確認できます！

In [ ]:
# 方法 A: 移動したいファイル名をリストで直接指定して承認
APPROVE_FILES = [
    # 例: "maniol-elvish-v4-mylora_00123_.png",
]

if APPROVE_FILES:
    move_df = suspects_df.copy()
    move_df["approve_move"] = move_df["file_name"].isin(APPROVE_FILES)
    apply_relabel_moves(move_df, dataset_root=DATASET_ROOT, dry_run=True)   # ← 確認後 False に変更
else:
    # 方法 B: 手動編集した audit_suspects.csv の approve_move==True 行を移動
    apply_relabel_moves(SUSPECTS_CSV, dataset_root=DATASET_ROOT, dry_run=True)

In [ ]:
# ==========================================
# 監査レポート＆整理済みデータセットの Drive 保存
# ==========================================
DRIVE_OUTPUT_DIR = f"/content/drive/MyDrive/Colab Notebooks/snapshot-inspector/{CHARACTER_NAME}/audit_001"

save_results_to_drive(
    DRIVE_OUTPUT_DIR,
    file_paths=[
        AUDITED_CSV,
        SUSPECTS_CSV,
        os.path.join(CACHE_DIR, "features_cache.npz"),
        os.path.join(CACHE_DIR, "features_table.csv"),
    ],
)

# 整理済みフォルダ全体を ZIP で Drive へバックアップしたい場合はコメント解除
# archive_folder_to_drive(
#     src_dir=DATASET_ROOT,
#     drive_output_dir=DRIVE_OUTPUT_DIR,
#     zip_name=f"{CHARACTER_NAME}_curated_v1_cleaned.zip",
#     local_work_dir=AUDIT_OUTPUT_DIR,
#     compress=False,
# )

### VS Code 上でのランタイム終了手順
作業が終わったら、GPUランタイムの課金・リソース消費を防ぐため、必ずランタイムを切断・削除してください。

1. VS Code 上部のコマンドパレット（`Ctrl + Shift + P` / macOS は `Cmd + Shift + P`）を開く
2. **`Colab: Remove Server`** を入力して実行する
3. 確認ダイアログが出たら対象の Colab ランタイムを削除して終了する